In [1]:
import os
import kagglehub
import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.utils.class_weight import compute_class_weight
from pathlib import Path
from random import sample
import seaborn as sns
import tkinter as tk
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
from PIL import Image, ImageTk

In [2]:
os.environ["KAGGLEHUB_CACHE"] = "data/kagglehub_cache"

path = kagglehub.dataset_download(
    "preetviradiya/brian-tumor-dataset"
)
print(path)

data/kagglehub_cache\datasets\preetviradiya\brian-tumor-dataset\versions\1


In [3]:
def load_images_from_folders(root_path):
    images = []
    labels = []

    class_map = {
        "Brain Tumor": 1,
        "Healthy": 0
    }

    for class_name, label in class_map.items():
        class_path = os.path.join(root_path, class_name)
        for file in os.listdir(class_path):
            if file.lower().endswith(('.jpg', '.jpeg', '.png', '.tif', '.tiff')):
                img_path = os.path.join(class_path, file)

                try:
                    img = Image.open(img_path)
                    img = img.convert("RGB")
                    img = img.resize((128, 128))
                    img_array = np.array(img)
                    if img_array.dtype == np.uint16:
                        img_array = img_array.astype(np.float32) / 65535.0
                    else:
                        img_array = img_array.astype(np.float32) / 255.0

                    images.append(img_array)
                    labels.append(label)

                except Exception as e:
                    print(f"Ошибка: {file} -> {e}")

    return np.array(images), np.array(labels)

In [4]:
root_path = Path(path) / "Brain Tumor Data Set" / "Brain Tumor Data Set"
X, Y = load_images_from_folders(
    root_path=root_path
)

print(X.shape)
print(Y.shape)

sss = StratifiedShuffleSplit(n_splits=1, train_size=0.8, random_state=42)
train_idx, test_idx = next(sss.split(X, Y))

x_train, x_test = X[train_idx], X[test_idx]
y_train, y_test = Y[train_idx].astype(np.float32), Y[test_idx].astype(np.float32)

(4600, 128, 128, 3)
(4600,)


In [5]:
def check_class_balance(labels, name):
    unique, counts = np.unique(labels, return_counts=True)

    print(f"\n{name} распределение:")
    for u, c in zip(unique, counts):
        class_name = "Healhty (0)" if u == 0 else "Brain Tumor (1)"
        percent = (c / len(labels)) * 100
        print(f"{class_name}: {c} изображений ({percent:.2f}%)")

check_class_balance(y_train, "Train выборка")
check_class_balance(y_test, "Test выборка")


Train выборка распределение:
Healhty (0): 1670 изображений (45.38%)
Brain Tumor (1): 2010 изображений (54.62%)

Test выборка распределение:
Healhty (0): 417 изображений (45.33%)
Brain Tumor (1): 503 изображений (54.67%)


In [6]:
# описание модели
model_max = Sequential([
    Input(shape=(128,128,3)),

    Conv2D(32, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

In [7]:
class_weights = compute_class_weight('balanced',
                                     classes=np.unique(y_train),
                                     y=y_train)
class_weight_dict = {0: class_weights[0], 1: class_weights[1]}

# сборка модели
model_max.compile(optimizer='adam',
                  loss='binary_crossentropy',
                  metrics=['accuracy'])

# обучение модели
model_max.fit(x_train, y_train, batch_size=16,
              epochs=5,
              validation_data=(x_test, y_test),
              class_weight=class_weight_dict)

Epoch 1/5
230/230 ━━━━━━━━━━━━━━━━━━━━ 11s 42ms/step - accuracy: 0.6978 - loss: 0.5581 - val_accuracy: 0.8457 - val_loss: 0.3429
Epoch 2/5
230/230 ━━━━━━━━━━━━━━━━━━━━ 10s 44ms/step - accuracy: 0.8451 - loss: 0.3460 - val_accuracy: 0.9152 - val_loss: 0.2159
Epoch 3/5
230/230 ━━━━━━━━━━━━━━━━━━━━ 11s 49ms/step - accuracy: 0.9117 - loss: 0.2199 - val_accuracy: 0.9478 - val_loss: 0.1497
Epoch 4/5
 60/230 ━━━━━━━━━━━━━━━━━━━━ 7s 45ms/step - accuracy: 0.9305 - loss: 0.1490

KeyboardInterrupt: 

In [ ]:
# проверка модели на тестовых данных
predictions_max = model_max.predict(x_test)
binary_predictions_max = (predictions_max > 0.5).astype(int).flatten()
accuracy_max = np.mean(y_test == binary_predictions_max)
print(f'Точность предсказания на тестовых данных : {accuracy_max * 100:.5f}%')

In [ ]:
cm_max = confusion_matrix(y_test, binary_predictions_max)

print("\nМетрики классификации:")
print(classification_report(
    y_test,
    binary_predictions_max,
    target_names=['Healthy', 'Brain Tumor']
))

plt.figure(figsize=(6,5))

sns.heatmap(
    cm_max,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Healthy", "Brain Tumor"],
    yticklabels=["Healthy", "Brain Tumor"],
    cbar=True
)

plt.title("Матрица ошибок (max пулинг)")
plt.xlabel("Предсказанный класс")
plt.ylabel("Истинный класс")

plt.tight_layout()
plt.show()